# RNAP Hack 2026 — Entrenamiento

## Track 3: Voice Note to Order

**Equipo:** RNAP-HACK-EQUIPO1 · **Producto:** Receipt

Este cuaderno es el reporte del curso. No contiene lógica propia: el código vive en `receipt/` y las cifras se leen de `results/`. Si una cifra aún no existe, la celda lo dice.

---

## 1. Problema y datos

**Reto elegido:** Track 3 — Voice Note to Order

**Cliente:** Ferretería Los Nogales (ficticia), entrega en obras del área de Monterrey.

**Descripción del problema:**
Convertir notas de voz en español (WhatsApp) en un ticket de pedido con cinco campos: nombre del cliente, producto, cantidad, medida y hora de entrega. El personal captura los pedidos a mano y se equivoca en cantidades y medidas (por ejemplo, pedir cable del doce y recibir calibre 10).

**Campos del ticket:** cliente, producto (`producto_id`), cantidad, medida, hora_entrega.

**Fuente de los datos:**
- Catálogo de productos (`data/catalog.csv`): 42 productos con categoría, sinónimos, unidad y medida.
- Notas de voz grabadas por el equipo: 30 audios (jack 8, jesus 7, william 8, carlos 7), con pedidos actuados. Tres eran copias idénticas de otra nota y se quitaron para que una misma nota no quede en entrenamiento y en prueba, así que quedan **27 notas** en `data/notes.csv`. El audio no sube al repo (`data/audio/`).
- La transcripción de referencia es un borrador hecho con `whisper-large-v3`, un modelo distinto de los que se evalúan, para no comparar un modelo contra sí mismo. El equipo debe revisarla escuchando cada audio. El nivel de ruido y la velocidad son estimaciones automáticas.
- Casi todas las notas piden varios productos y el ticket tiene uno solo. El ticket correcto es el **primer producto mencionado**, con su cantidad. El cliente se anota sin título ("el ingeniero Carlos Ruiz" se anota como `Carlos Ruiz`).
- Cualquier texto sintético que se use para desarrollar reglas va etiquetado como sintético y no sustituye al audio ni permite medir WER.

**División de datos:** 60 % entrenamiento, 20 % validación y 20 % prueba, con semilla 42 (`python -m receipt split` escribe `data/splits.csv`). Las notas se ordenan por persona y por nivel de ruido, y las proporciones se cuentan sobre todas juntas: quedan 16 de entrenamiento, 6 de validación y 5 de prueba, con las cuatro personas y los tres niveles de ruido en la prueba. La división se hizo antes de evaluar y la prueba se usa solo para las métricas finales. Los modelos de la Fase 1 no se entrenan: el conjunto de entrenamiento sirve para ajustar las reglas.

Las celdas siguientes muestran el catálogo y la división.

In [1]:
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from receipt.catalog import read_products

products = read_products(ROOT / "data" / "catalog.csv")
print(f"Productos en el catálogo: {len(products)}")
for category, count in sorted(Counter(p.categoria for p in products).items()):
    print(f"  {category}: {count}")

Productos en el catálogo: 42
  Acero: 6
  Cable: 7
  Electrico: 4
  Fijaciones: 6
  Materiales: 7
  Pintura: 2
  Plomeria: 5
  Tubo: 5


In [2]:
import pandas as pd

notes_path = ROOT / "data" / "notes.csv"
splits_path = ROOT / "data" / "splits.csv"
notes = pd.read_csv(notes_path, dtype=str, keep_default_na=False) if notes_path.is_file() else pd.DataFrame()
print(f"Notas en data/notes.csv: {len(notes)}")
if notes.empty:
    print("Pendiente: el equipo aún no agrega notas de voz (F1.3a).")
elif not splits_path.is_file():
    print("Pendiente: falta crear la división con `python -m receipt split`.")
else:
    print(pd.read_csv(splits_path, dtype=str)["split"].value_counts().to_string())

Notas en data/notes.csv: 27
split
train    16
val       6
test      5


---

## 2. Representación del texto

**Entrada:** audio de nota de voz en español (`.wav`, `.mp3`, `.m4a`, `.ogg`, `.opus`, `.webm`), decodificado con PyAV a 16 kHz mono.

**Transcripción:** Whisper (Transformer encoder-decoder, vía `transformers`) convierte el audio a texto con idioma español y tarea `transcribe` forzados. Los audios de más de 30 s usan la generación long-form de Whisper.

**Normalización:** minúsculas, sin acentos ni puntuación y números en dígitos (`cincuenta y dos` pasa a `52`). Las fracciones tienen una sola forma (`tres cuartos`, `3 cuartos` y `3/4` pasan a `3/4`; `media` pasa a `1/2`). El mismo texto normalizado se usa para el WER, para comparar con los sinónimos del catálogo y para las reglas, de modo que referencia, hipótesis y catálogo coincidan.

**Extracción de campos:**
- Línea base: reglas con expresiones regulares sobre los sinónimos del catálogo, buscando primero el sinónimo más largo (`receipt.extraction.rules`). La medida sale del producto del catálogo, no de los números del texto: en "50 metros de cable del doce", 50 es la cantidad y 12 es el calibre.
- Preentrenado: modelo de lenguaje local (`qwen3:4b-instruct-2507-q4_K_M` con Ollama) que recibe el catálogo compacto y devuelve JSON con id de producto, cliente, cantidad y hora (`receipt.extraction.llm`). Si Ollama falla se usan las reglas y el resultado queda marcado como `used_fallback`.

**Emparejamiento con el catálogo:** el producto sale de un sinónimo del catálogo o del id que devuelve el LLM. En el modo `solution`, si el LLM no da producto, se empareja por similitud de embeddings (`paraphrase-multilingual-MiniLM-L12-v2`).

**Campos dudosos:** el ticket marca en `review_fields` todo campo vacío, la `medida` si el producto existe en otras medidas de su familia y no se oyó claramente, y la `hora_entrega` si se dijo sin mañana/tarde/noche y no es inequívoca.

**Justificación:** Whisper es un modelo preentrenado que funciona sin ajuste en español. Las reglas con regex son una línea base rápida, interpretable y sin red; el LLM extrae JSON estructurado y se espera que maneje mejor la variabilidad del lenguaje hablado (hipótesis por medir). Ambos se comparan sobre las mismas notas de prueba con WER y porcentaje de campos correctos.

---

## 3. Modelo base y preentrenado sin ajuste

### 3.1 Modelo base: Whisper small + reglas con expresiones regulares

- Transcripción: `openai/whisper-small` con `transformers`.
- Extracción: reglas con expresiones regulares (`receipt.extraction.rules`).
- Sin fine-tuning.

### 3.2 Modelo preentrenado (sin ajuste): Whisper medium + extracción con modelo de lenguaje

- Transcripción: `openai/whisper-medium` con `transformers`.
- Extracción: `qwen3:4b-instruct-2507-q4_K_M` local con Ollama, `temperature` 0 y salida JSON con esquema.
- Sin fine-tuning, usado tal como viene.

Ambos modelos se evalúan sobre las mismas notas de prueba, con `python -m receipt evaluate --mode baseline --split test` y `--mode pretrained`.

### Resultados comparativos

La tabla se llena sola desde `results/` (un archivo `<modo>_<división>.json` por corrida). Reporta WER, % de campos correctos en total y por campo, tiempos por nota (media y p95), tiempo de carga del modelo aparte, dispositivo y número de notas que usaron el respaldo de reglas.

In [3]:
from receipt.evaluation import comparison_table

try:
    print(comparison_table(ROOT / "results"))
except FileNotFoundError:
    print("Pendiente: aún no hay resultados en results/. Se generan con `python -m receipt evaluate`.")

| Modo | División | Notas | WER | Campos OK | cliente | producto_id | cantidad | medida | hora_entrega | Transcripción (s, media) | Extracción (s, media) | Total (s, p95) | Carga (s) | Con fallback | Dispositivo |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | test | 5 | 18.3% | 60.0% | 80.0% | 20.0% | 80.0% | 20.0% | 100.0% | 4.68 | 0.01 | 7.55 | 40.5 | 0 | cuda |
| pretrained | test | 5 | 12.7% | 76.0% | 60.0% | 100.0% | 40.0% | 100.0% | 80.0% | 8.82 | 2.84 | 18.11 | 58.0 | 0 | cuda |


**Conclusión breve:** en las 5 notas de prueba, el modelo preentrenado (Whisper medium + qwen3:4b) es más preciso. Su WER es de 12.7 % contra 18.3 % de la línea base, y acierta el 76 % de los campos contra el 60 %. La mayor diferencia está en producto y medida, con 100 % contra 20 %. Las reglas solo encuentran el producto si la transcripción contiene un sinónimo exacto del catálogo, y Whisper small escribió "blog de concreto", "impermeabilidad santa" y "cable THN". En cambio, las reglas aciertan más en cliente (80 % contra 60 %), cantidad (80 % contra 40 %) y hora (100 % contra 80 %). El modelo de lenguaje dejó títulos en el nombre ("don José", "ARKY Benítez"), tomó la cantidad de otro producto y convirtió "10 p.m.", que Whisper medium escuchó así, en 22:00. La línea base también es más rápida: 4.69 s por nota en promedio contra 11.66 s, y 7.55 s contra 18.11 s en el p95, en la RTX 5050.

Estas cifras son preliminares. La prueba tiene solo 5 notas, así que cada nota vale 20 puntos en cada campo, y la transcripción de referencia todavía no se revisa escuchando el audio. Para la Fase 2 conviene combinar lo mejor de cada uno: el modelo de lenguaje para producto y medida, y las reglas para hora y cantidad.

### 3.3 Configuración documentada

Los valores salen de `receipt/config.py`. El efecto de cada parámetro sobre WER, % de campos correctos y tiempo está **pendiente de medir**: no hay corridas con audio real.

| Parámetro | Base | Preentrenado | Solución (modo `solution`) | Efecto observado |
|---|---|---|---|---|
| Modelo de Whisper | `whisper-small` | `whisper-medium` | `whisper-small` | pendiente de medir |
| Idioma y tarea | `es`, `transcribe` | `es`, `transcribe` | `es`, `transcribe` | no se varía |
| Prompt inicial con vocabulario del catálogo | no | no | sí | pendiente de medir |
| Extractor | reglas | LLM | LLM | pendiente de medir |
| Prompt de extracción | no aplica | catálogo compacto + reglas de calibre y hora | igual | pendiente de medir |
| Emparejamiento por embeddings | no | no | sí | pendiente de medir |
| Umbral de similitud | no aplica | no aplica | 0.6 (valor inicial sin calibrar) | pendiente de medir |
| Semilla | 42 | 42 | 42 | no aplica |
| División train/val/test | 60/20/20 | 60/20/20 | 60/20/20 | no aplica |

No hay fine-tuning, así que tasa de aprendizaje y tamaño de lote no aplican.

---

## 4. Solución con Transformer

### Modelo y justificación

El modo `solution` del pipeline encadena tres pasos:

1. **Transcripción** con Whisper small (Transformer encoder-decoder), forzando idioma español y usando como prompt inicial el vocabulario del catálogo (`prompt_ids`).
2. **Extracción** de cliente, producto, cantidad y hora con el modelo de lenguaje local, que devuelve JSON. Si Ollama falla se usan las reglas y el ticket queda marcado como respaldo; nunca se oculta.
3. **Emparejamiento con el catálogo** por embeddings de sentencias (`paraphrase-multilingual-MiniLM-L12-v2`) cuando el LLM no entrega un producto válido, con un umbral de similitud configurable, para acercar frases como "cable del doce" a `CAB-003`.

Justificación: Whisper es un Transformer encoder-decoder, por lo que su mapa de cross-attention permitirá explicar a qué parte del audio atiende el decoder al escribir cada palabra (Fase 3). La extracción guiada por el catálogo busca reducir el error real de la ferretería, la confusión de cantidad y medida; que lo logre es una hipótesis que falta medir.

### Hipótesis sobre los parámetros clave

Son hipótesis, no resultados. Ningún efecto está medido todavía.

| Parámetro | Hipótesis | Efecto observado |
|---|---|---|
| Tamaño de Whisper | Un modelo mayor podría bajar el WER a costa de más tiempo y memoria. | pendiente de medir |
| Prompt inicial (vocabulario del catálogo) | Podría reducir errores de producto al orientar a Whisper hacia los términos del catálogo. | pendiente de medir |
| Prompt de extracción | Un prompt que aclare que el calibre no es la cantidad podría reducir confusiones. | pendiente de medir |
| Umbral de similitud | Un umbral alto daría menos falsos positivos y más productos sin reconocer; uno bajo, lo contrario. | pendiente de medir |

---

## 5. Evaluación y errores

### Métricas en el conjunto de prueba

Las mismas métricas de la Fase 1 sobre la prueba (división hecha antes de evaluar, la prueba solo para métricas finales):

- **WER** de la transcripción, con referencia e hipótesis normalizadas igual.
- **Porcentaje de campos correctos** del ticket (cliente, producto, cantidad, medida, hora de entrega), en total y por campo. El cliente se compara sin distinguir mayúsculas ni acentos, la cantidad como número y la hora como HH:MM, así que `50` y `cincuenta` valen lo mismo.
- **Tiempo de procesamiento** por nota (transcripción y extracción, media y p95). El tiempo de carga del modelo se excluye y se reporta aparte, tras una nota de calentamiento.

### Tabla comparativa de los tres modos

La tabla incluye todos los modos que tengan resultados guardados en `results/`; los modos sin corrida aparecen cuando existan.

In [4]:
from receipt.evaluation import comparison_table

try:
    print(comparison_table(ROOT / "results"))
except FileNotFoundError:
    print("Pendiente: aún no hay resultados en results/. Se generan con `python -m receipt evaluate`.")

| Modo | División | Notas | WER | Campos OK | cliente | producto_id | cantidad | medida | hora_entrega | Transcripción (s, media) | Extracción (s, media) | Total (s, p95) | Carga (s) | Con fallback | Dispositivo |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | test | 5 | 18.3% | 60.0% | 80.0% | 20.0% | 80.0% | 20.0% | 100.0% | 4.68 | 0.01 | 7.55 | 40.5 | 0 | cuda |
| pretrained | test | 5 | 12.7% | 76.0% | 60.0% | 100.0% | 40.0% | 100.0% | 80.0% | 8.82 | 2.84 | 18.11 | 58.0 | 0 | cuda |


### Gráfica

Pendiente: se generará a partir de `results/` (% de campos correctos y tiempo medio por modo) cuando haya resultados.

### Casos de error con causa propuesta

Pendiente. Se llenan con notas reales de `results/<modo>_<división>_notes.csv`, donde cada nota con algún campo incorrecto trae su transcripción, el ticket predicho y los campos fallidos. Se necesitan 5 o más casos y la causa es una propuesta del equipo.

| Caso | Transcripción obtenida | Ticket predicho | Ticket esperado | Causa propuesta |
|---|---|---|---|---|
| 1 | pendiente | pendiente | pendiente | pendiente |
| 2 | pendiente | pendiente | pendiente | pendiente |
| 3 | pendiente | pendiente | pendiente | pendiente |
| 4 | pendiente | pendiente | pendiente | pendiente |
| 5 | pendiente | pendiente | pendiente | pendiente |

---

## 6. Comparación final y límites

Pendiente de la Fase 3 (`v1.0`). Contendrá:

- Tabla final de la línea base contra la solución, leída de `results/`.
- El elemento de atención: mapa de cross-attention de Whisper en un acierto y en un error.
- Límites del sistema, con ejemplos reales de las notas nuevas y ruidosas de la prueba del juez.